# 🏆 Gemma Developer Agent: Clean-Bench (Rich) Rigorous A/B Testing Suite

> ### 📌 **Notebook Version: v8.0-clean-rich (Updated: 2026-11-08)**
> - **タスクセットの正常化 (Textualize/rich)**: 環境不備で 100% 失敗する壊れたタスク (FastAPI 27問) を排除し、コミュニティ監査で健全性 39/48 (約81%) が実証された **クリーンな Rich 実タスク** を採用。
> - **3 大仮説の科学的 A/B 検証**:
>   1. **Variant A (V9.2 Baseline / LB 0.17 実績)**: 思考制約なし・波括弧なし・Call 8 強制編集
>   2. **Variant B (V14 Real-Proof / LB 0.12 実績)**: 思考 2 文制約あり・波括弧 One-Shot あり
>   3. **Variant C (V15 候補 / Loop-Breaker)**: 思考制約なし・波括弧なし・**`edit_file` 失敗時の Python 即時迂回ルール** 導入
> - **波括弧 `{` の ADK テンプレート破壊テスト**: ハーネス側で波括弧がどう処理されるかを直接実機検証。

In [ ]:
# 1. 【A100 必須ハードウェア・ゲート】 & 環境初期化 (A100 以外は 0.5 秒で即時中断)
import os
import sys
import time
import json
import re
import gc
import subprocess
from pathlib import Path
import torch

# ==============================================================================
# ⚙️ 設定フラグ:
#   REQUIRE_A100 = True  -> A100 (VRAM >= 35GB) 以外が割り当てられた場合、
#                           パッケージ導入やモデルDLを行わず 0.5 秒で即時中断します。
#   AUTO_DISCONNECT_NON_A100 = False -> True にすると A100 以外のランタイムを自動切断(unassign)します。
# ==============================================================================
REQUIRE_A100 = True
AUTO_DISCONNECT_NON_A100 = False

if not torch.cuda.is_available():
    raise SystemExit("❌ [ABORT] GPU が割り当てられていません (CPU ランタイム)。実行を即時中断しました。")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
device_name = torch.cuda.get_device_name(0)
print(f"[Hardware Gate] 検出 GPU: {device_name} (総 VRAM: {vram_gb:.2f} GB)")

if REQUIRE_A100 and ("A100" not in device_name and vram_gb < 35.0):
    msg = (
        f"❌ [ABORT: 非 A100 検出] 現在の割り当ては {device_name} ({vram_gb:.2f} GB) です。\n"
        f"   A100 ではないため、無駄なパッケージ導入・モデルDL・CU消費を防ぐためセル1で即時中断しました。\n"
        f"   (※ L4 のまま 27B-4bit で実行したい場合は、セル1先頭の REQUIRE_A100 = False に変更してください)"
    )
    print(msg)
    if AUTO_DISCONNECT_NON_A100:
        try:
            from google.colab import runtime
            print("[Hardware Gate] 非 A100 ランタイムを自動解除 (runtime.unassign) します...")
            runtime.unassign()
        except Exception:
            pass
    raise SystemExit(msg)

print(f"✅ [Hardware Gate PASS] A100 クラスの GPU ({device_name}, {vram_gb:.2f} GB) を確認しました。セットアップを続行します...")
print("[Setup] 依存パッケージをインストール中 (transformers, bitsandbytes, compressed-tensors, pytest, etc.)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "transformers", "accelerate", "bitsandbytes", "compressed-tensors",
     "sentencepiece", "pytest", "pytest-asyncio", "requests", "gitpython", "kaggle", "tabulate", "rich"],
    check=True
)
print("[Setup] ✅ パッケージ導入完了。")

# Kaggle API による最新提出状況チェック
try:
    from google.colab import userdata
    k_user = userdata.get("KAGGLE_USERNAME")
    k_key = userdata.get("KAGGLE_KEY")
    if k_user and k_key:
        os.environ["KAGGLE_USERNAME"] = k_user
        os.environ["KAGGLE_KEY"] = k_key
        res = subprocess.run(["kaggle", "competitions", "submissions", "-c", "gemma-4-developer-agent"], capture_output=True, text=True)
        print("\n[Kaggle Submissions Status]:")
        for line in res.stdout.split("\n")[:5]:
            print("  " + line)
except Exception as e:
    print(f"[Kaggle Notice] Kaggle API チェック スキップ: {e}")


In [ ]:
# 2. サンドボックス環境の準備 (Textualize/rich クリーンリポジトリ)
SANDBOX_DIR = "/content/mini_kaggle_sandbox"
REPO_DIR = os.path.join(SANDBOX_DIR, "rich_repo")
os.makedirs(SANDBOX_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    print("[Sandbox] Textualize/rich 公式リポジトリをクローン中...")
    !git clone --quiet https://github.com/Textualize/rich.git {REPO_DIR}
else:
    print("[Sandbox] 既存の Rich リポジトリを使用します。")

print("[Sandbox] Rich のテスト実行環境をセットアップ中...")
!pip install --quiet typing_extensions pygments pytest pytest-cov
print("[Sandbox] ✅ Rich サンドボックス準備完了。")


In [ ]:
# 3. 本番公式 31B モデルのロード (A100-40GB 解凍 OOM 完全回避 max_memory 5000MiB 仕様)
import gc
import os
import sys
import time
import traceback
import torch
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def force_free_vram():
    for var_name in ["model", "tokenizer"]:
        if var_name in globals():
            del globals()[var_name]
    for tb_attr in ["last_traceback", "last_value", "last_type"]:
        if hasattr(sys, tb_attr):
            obj = getattr(sys, tb_attr)
            if tb_attr == "last_traceback" and obj is not None:
                try:
                    traceback.clear_frames(obj)
                except Exception:
                    pass
            setattr(sys, tb_attr, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

force_free_vram()

hf_token = None
try:
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
free_gb = (torch.cuda.mem_get_info()[0] / 1e9) if torch.cuda.is_available() else 0
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
cpu_ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9 if hasattr(os, "sysconf") else 50.0
print(f"[Hardware] GPU: {device_name} (Total VRAM: {vram_gb:.2f} GB / Free: {free_gb:.2f} GB) | System RAM: {cpu_ram_gb:.1f} GB")

model = None
tokenizer = None
MODEL_ID = ""
LOAD_NOTES = ""

PRIMARY_ID = "google/gemma-4-31B-it-qat-w4a16-ct"

# ------------------------------------------------------------------------------
# 🎯 A100-40GB / A100-80GB に応じた max_memory の動的最適化
# ⚠️ 実測根拠: 31B-QAT は初回 forward の Decompressing で GPU 上のレイヤーが約 4.49 倍に膨張する。
#   - A100-80GB (VRAM >= 70GB): max_memory=None で全量を GPU 0 に配置 (展開後約60GBでも85GBに余裕)
#   - A100-40GB (30GB <= VRAM < 70GB): 初期 GPU 配置を 5000MiB (~28% レイヤー) に絞り、
#     4.49倍解凍後ピークを約 22.5 GiB / 39.5 GiB に抑えて 15 GiB 以上の安全マージンを確保！
#   - L4 (VRAM < 30GB): 初期 GPU 配置を 3000MiB に絞り、解凍後ピークを約 13.5 GiB / 23.7 GiB に抑制。
# ------------------------------------------------------------------------------
if vram_gb >= 70.0:
    max_mem = None
    mem_label = "全量 GPU 0 (80GB モード)"
elif vram_gb >= 30.0:
    max_mem = {0: "5000MiB", "cpu": "75GiB"}
    mem_label = "GPU 5000MiB + CPU 75GiB (A100-40GB 安全解凍モード: 安全マージン 15GB)"
else:
    max_mem = {0: "3000MiB", "cpu": "75GiB"}
    mem_label = "GPU 3000MiB + CPU 75GiB (L4 安全解凍モード: 安全マージン 10GB)"

print(f"[Model] 本番公式 31B モデル ({PRIMARY_ID}) をロード中... [{mem_label}]")
t_start = time.time()
try:
    tokenizer = AutoTokenizer.from_pretrained(PRIMARY_ID, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        PRIMARY_ID,
        device_map="auto",
        max_memory=max_mem,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        token=hf_token
    )
    MODEL_ID = PRIMARY_ID
    LOAD_NOTES = f"31B-QAT on {device_name} ({vram_gb:.1f}GB, {mem_label})"
    print(f"[Model] [OK] 31B-QAT ロード成功！ 所要時間: {time.time() - t_start:.1f}s")
# ------------------------------------------------------------------------------
# 🛡️ Accelerate / Compressed-Tensors _hf_hook 互換性パッチ
# Gemma4ForConditionalGeneration 等のルートモジュールで _hf_hook が未定義による
# AttributeError を物理的に根絶する安全ガード
# ------------------------------------------------------------------------------
class SafeHfHook:
    def __init__(self, device="cuda:0"):
        self.weights_map = {}
        self.execution_device = device
        self.io_same_device = True
        self.offload = False
        self.place_submodules = False

    def __getattr__(self, name):
        return None

    def __bool__(self):
        return False

hook_dev = "cuda:0" if torch.cuda.is_available() else "cpu"
for m in model.modules():
    if "_hf_hook" not in m.__dict__:
        m._hf_hook = SafeHfHook(hook_dev)

print("[Model] [OK] _hf_hook 互換性パッチを全モジュールに適用完了。")

except Exception as e:
    LOAD_NOTES = f"31B load failed ({type(e).__name__}: {str(e)[:80]})"
    print(f"[Model Error] 31B ロード失敗: {e}")
    raise e

alloc = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"[Model Ready] 確定稼働モデル: {MODEL_ID} | 初期 VRAM: {alloc:.2f} GB / 空き: {vram_gb - alloc:.1f} GB | Notes: {LOAD_NOTES}")

def call_llm(messages, temperature=0.2, max_new_tokens=1024):
    normalized = []
    sys_prefix = ""
    for m in messages:
        role = m["role"]
        if role == "system":
            sys_prefix += m["content"].strip() + "\n\n"
            continue
        r = "assistant" if role in ["assistant", "model"] else "user"
        content = m.get("content", "").strip()
        if not content:
            content = "(empty)"
        if r == "user" and sys_prefix:
            content = sys_prefix + content
            sys_prefix = ""
        if not normalized:
            normalized.append({"role": r, "content": content})
        elif normalized[-1]["role"] == r:
            normalized[-1]["content"] += "\n\n" + content
        else:
            normalized.append({"role": r, "content": content})
            
    if not normalized and sys_prefix:
        normalized.append({"role": "user", "content": sys_prefix})
        
    try:
        prompt_text = tokenizer.apply_chat_template(normalized, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt_text = ""
        for m in normalized:
            tag = "model" if m["role"] == "assistant" else "user"
            prompt_text += f"<start_of_turn>{tag}\n{m['content']}<end_of_turn>\n"
        prompt_text += "<start_of_turn>model\n"
        
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0" if torch.cuda.is_available() else "cpu")
    input_len = inputs["input_ids"].shape[1]
    
    t0 = time.time()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0.0),
            temperature=max(temperature, 1e-4),
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - t0
    resp = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    del inputs, outputs
    return resp, elapsed

print("[Engine] [OK] 推論関数 call_llm 準備完了。")



In [ ]:
# 4. 実環境ツール実行エンジン & 万能マルチ形式パーサー (誤爆防止・プレースホルダ除外強化版)
import ast
import re
import json
import os
import subprocess

PLACEHOLDER_VALUES = {"...", "target.py", "tests/test_target.py", "tests/path/test_x.py", "path/to/file.py"}

def is_placeholder_call(name, args):
    if not isinstance(args, dict):
        return False
    if name in ["read_file", "edit_file"]:
        p = str(args.get("path", "")).strip()
        if not p or p in PLACEHOLDER_VALUES or "..." in p:
            return True
    if name == "edit_file":
        o = str(args.get("old_string", "")).strip()
        if o in PLACEHOLDER_VALUES:
            return True
    if name == "run_command":
        c = str(args.get("command", "")).strip()
        if not c or c in PLACEHOLDER_VALUES or "'KEYWORD'" in c or "'SYMBOL'" in c or "'TOKEN'" in c:
            return True
    return False

def parse_tool_call(response_text):
    """モデルの出力から実際のツール呼び出しを抽出する (例示プレースホルダや計画末尾の submit_patch 誤爆を完全排除)"""
    candidates = []

    # 1. <tool_call> または JSON ブロック
    for pat in [r'<tool_call>\s*(\{.*?\})\s*</tool_call>', r'```(?:json)?\s*(\{.*?\})\s*```']:
        for m in re.finditer(pat, response_text, re.DOTALL):
            try:
                data = json.loads(m.group(1).strip())
                name = data.get("name") or data.get("tool") or ""
                args_val = data.get("arguments") or data.get("args") or {}
                if isinstance(args_val, str):
                    try: args_val = json.loads(args_val)
                    except Exception: args_val = {"raw_input": args_val}
                if name and not is_placeholder_call(name, args_val):
                    candidates.append(({"name": name, "arguments": args_val}, {"has_tool": True, "parse_ok": True}))
            except Exception:
                pass

    # 2. コードブロック抽出
    code_blocks = re.findall(r'```(?:python|bash|sh)?\s*\n(.*?)```', response_text, re.DOTALL)
    if not code_blocks:
        code_blocks = [response_text]

    for code in code_blocks:
        code = code.strip()
        if not code:
            continue

        # AST 関数呼び出し解析
        try:
            tree = ast.parse(code)
            for node in ast.walk(tree):
                if isinstance(node, ast.Call):
                    fn_name = None
                    if isinstance(node.func, ast.Name):
                        fn_name = node.func.id
                    elif isinstance(node.func, ast.Attribute):
                        fn_name = node.func.attr
                    if fn_name in ["edit_file", "read_file", "run_command", "get_status", "submit_patch"]:
                        args = {}
                        for kw in node.keywords:
                            if isinstance(kw.value, ast.Constant):
                                args[kw.arg] = str(kw.value.value)
                        pos_names = {
                            'edit_file': ['path', 'old_string', 'new_string'],
                            'read_file': ['path', 'start_line', 'end_line'],
                            'run_command': ['command']
                        }
                        for i, a in enumerate(node.args):
                            if i < len(pos_names.get(fn_name, [])) and isinstance(a, ast.Constant):
                                args[pos_names[fn_name][i]] = str(a.value)
                        if "file_path" in args and "path" not in args:
                            args["path"] = args.pop("file_path")
                        if "filepath" in args and "path" not in args:
                            args["path"] = args.pop("filepath")
                        if is_placeholder_call(fn_name, args):
                            continue
                        if fn_name == "edit_file":
                            missing_old = "old_string" not in args or not args.get("old_string")
                            if args.get("path"):
                                candidates.append(({"name": fn_name, "arguments": args}, {"has_tool": True, "parse_ok": True, "missing_old_string": missing_old}))
                        else:
                            candidates.append(({"name": fn_name, "arguments": args}, {"has_tool": True, "parse_ok": True}))
        except Exception:
            pass

        # Regex フォールバック (複数行 edit_file 等で AST パースが失敗した場合)
        for fn_name in ["edit_file", "read_file", "run_command"]:
            if fn_name + "(" in code:
                if fn_name == "edit_file":
                    m_p = re.search(r'(?:path|file_path)\s*=\s*["\']([^"\']+)["\']', code)
                    m_o = re.search(r'old_string\s*=\s*["\']{1,3}([\s\S]*?)["\']{1,3}', code)
                    m_n = re.search(r'new_string\s*=\s*["\']{1,3}([\s\S]*?)["\']{1,3}', code)
                    if m_p and m_o is not None:
                        args = {"path": m_p.group(1), "old_string": m_o.group(1), "new_string": m_n.group(1) if m_n else ""}
                        if not is_placeholder_call("edit_file", args):
                            candidates.append(({"name": "edit_file", "arguments": args}, {"has_tool": True, "parse_ok": True}))
                elif fn_name == "read_file":
                    m_p = re.search(r'(?:path|file_path)\s*=\s*["\']([^"\']+)["\']', code)
                    if not m_p:
                        m_p = re.search(r'read_file\(\s*["\']([^"\']+)["\']', code)
                    if m_p:
                        args = {"path": m_p.group(1)}
                        m_s = re.search(r'start_line\s*=\s*(\d+)', code)
                        m_e = re.search(r'end_line\s*=\s*(\d+)', code)
                        if m_s: args["start_line"] = int(m_s.group(1))
                        if m_e: args["end_line"] = int(m_e.group(1))
                        if not is_placeholder_call("read_file", args):
                            candidates.append(({"name": "read_file", "arguments": args}, {"has_tool": True, "parse_ok": True}))
                elif fn_name == "run_command":
                    m_c = re.search(r'(?:command|cmd)\s*=\s*["\']{1,3}([\s\S]*?)["\']{1,3}', code)
                    if not m_c:
                        m_c = re.search(r'run_command\(\s*["\']([^"\']+)["\']', code)
                    if m_c:
                        args = {"command": m_c.group(1)}
                        if not is_placeholder_call("run_command", args):
                            candidates.append(({"name": "run_command", "arguments": args}, {"has_tool": True, "parse_ok": True}))

        # シェルコマンド直書きフォールバック
        first_line = code.splitlines()[0].strip() if code.splitlines() else ""
        if any(first_line.startswith(cmd) for cmd in ["pytest", "git ", "python ", "python3 ", "grep "]):
            args = {"command": first_line}
            if not is_placeholder_call("run_command", args):
                candidates.append(({"name": "run_command", "arguments": args}, {"has_tool": True, "parse_ok": True}))
        elif first_line.startswith("submit_patch"):
            candidates.append(({"name": "submit_patch", "arguments": {}}, {"has_tool": True, "parse_ok": True}))

    if not candidates:
        return None, {"has_tool": False, "error": "No recognized tool format"}

    # ⚠️ 重要: 1発話内に実アクション (run_command, read_file, edit_file) と submit_patch が混在している場合、
    # モデルが将来の計画として末尾に submit_patch() を書いただけなので、submit_patch 以外の先頭アクションを優先採用する！
    non_submit = [c for c in candidates if c[0]["name"] != "submit_patch"]
    if non_submit:
        return non_submit[0]
    return candidates[0]


def execute_tool(tool_name, arguments, repo_dir):
    """リポジトリサンドボックス上で実際のツールを実行する"""
    try:
        # もし run_command の中身が read_file(...) や edit_file(...) だった場合は自動救済ルーティング
        if tool_name == "run_command":
            cmd_str = str(arguments.get("command") or arguments.get("cmd") or "").strip()
            if cmd_str.startswith("read_file(") or cmd_str.startswith("edit_file("):
                rescued, _ = parse_tool_call(f"```python\n{cmd_str}\n```")
                if rescued and rescued["name"] in ["read_file", "edit_file"]:
                    tool_name = rescued["name"]
                    arguments = rescued["arguments"]

        def sanitize_path(raw_path):
            if not raw_path: return ""
            p = str(raw_path).strip().replace("'", "").replace('"', '')
            for pfx in ["/workspace/rich/", "/workspace/", "workspace/", "/"]:
                while p.startswith(pfx):
                    p = p[len(pfx):]
            target = os.path.join(repo_dir, p)
            if os.path.exists(target) and os.path.isfile(target):
                return target
            parts = p.split("/")
            if len(parts) >= 2 and parts[0] == parts[1]:
                dedup_p = "/".join(parts[1:])
                target_dedup = os.path.join(repo_dir, dedup_p)
                if os.path.exists(target_dedup) and os.path.isfile(target_dedup):
                    return target_dedup
            base = os.path.basename(p)
            if base:
                for root, _, files in os.walk(repo_dir):
                    if base in files and "tests" not in root:
                        return os.path.join(root, base)
                for root, _, files in os.walk(repo_dir):
                    if base in files:
                        return os.path.join(root, base)
            return target

        raw_path_val = arguments.get("path") or arguments.get("file_path") or arguments.get("filepath") or arguments.get("file") or ""
        if tool_name == "read_file":
            p = sanitize_path(raw_path_val)
            if not p or not os.path.exists(p) or os.path.isdir(p):
                return f"Error: File not found: {raw_path_val}"
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                lines = f.readlines()
            s = max(1, int(arguments.get("start_line", 1))) - 1
            e = min(len(lines), int(arguments.get("end_line", min(len(lines), s + 80))))
            out = [f"{i+1}: {lines[i]}" for i in range(s, e)]
            return "".join(out)[:3500]
            
        elif tool_name == "edit_file":
            p = sanitize_path(raw_path_val)
            if not p or not os.path.exists(p) or os.path.isdir(p):
                return f"Error: File not found: {raw_path_val}"
            old_str = str(arguments.get("old_string") or arguments.get("old_str") or "")
            new_str = str(arguments.get("new_string") or arguments.get("new_str") or "")
            if not old_str:
                return "Error: `old_string` is required and cannot be empty. Copy 3-5 exact lines from read_file."
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
            if old_str not in content:
                # 自動ヒント: ファイル内の近似行を探して行番号を教える
                first_token = old_str.strip().splitlines()[0].strip()[:25] if old_str.strip() else ""
                hint = ""
                if first_token:
                    matches = [f"Line {idx+1}: {line}" for idx, line in enumerate(content.splitlines()) if first_token in line]
                    if matches:
                        hint = "\nFound similar lines in file:\n" + "\n".join(matches[:5])
                return f"Error: `old_string` not found in {raw_path_val}. Call read_file({raw_path_val}, start_line, end_line) first.{hint}"
            content = content.replace(old_str, new_str, 1)
            with open(p, "w", encoding="utf-8") as f:
                f.write(content)
            return f"Successfully edited {raw_path_val}."
            
        elif tool_name == "run_command":
            cmd = str(arguments.get("command") or arguments.get("cmd") or "")
            if not cmd:
                return "Error: command is required."
            res = subprocess.run(cmd, shell=True, cwd=repo_dir, capture_output=True, text=True, timeout=30)
            out = (res.stdout + "\n" + res.stderr).strip()
            return out[:2500] if out else "(No output, exit code: " + str(res.returncode) + ")"
            
        elif tool_name == "get_status":
            res = subprocess.run(["git", "status", "--short"], cwd=repo_dir, capture_output=True, text=True)
            return res.stdout.strip() if res.stdout.strip() else "No changes."
            
        elif tool_name == "submit_patch":
            return "SUBMIT_PATCH_CALLED"
            
        else:
            return f"Error: Unknown tool '{tool_name}'."
    except Exception as e:
        return f"Execution Error in {tool_name}: {e}"

print("[Engine] [OK] 万能マルチ形式パーサー (誤爆防止・プレースホルダ除外・自動ヒント付き) 準備完了。")


In [ ]:
# 5. 【検証 1】波括弧 `{` の Google ADK / テンプレートエンジン破壊直接テスト
print("="*75)
print("🔬 検証 1: プロンプト内の波括弧 { ... } がテンプレートエンジンに与える影響")
print("="*75)

prompt_with_curly = '''System instruction: Use tool format:
<tool_call>
{"name": "edit_file", "arguments": {"path": "rich/console.py", "old_string": "x = 1", "new_string": "x = 2"}}
</tool_call>
'''

# 1. Python str.format() テスト (ADK 内部で使われる場合)
try:
    res = prompt_with_curly.format(issue="dummy issue")
    print("  [str.format] ✅ 正常レンダリング")
except Exception as e:
    print(f"  [str.format] ❌ 致命的例外検知: {type(e).__name__} ({e})")
    print("    ➔ 根拠実証: プロンプト内に波括弧があると、str.format を使うハーネスは KeyError でクラッシュします！")

# 2. string.Template テスト
from string import Template
try:
    t = Template(prompt_with_curly)
    res2 = t.safe_substitute(issue="dummy issue")
    print("  [string.Template] ✅ 正常 (safe_substitute)")
except Exception as e:
    print(f"  [string.Template] ❌ 例外: {e}")

print("="*75)


In [ ]:
# 6. 【次世代 A/B 実機検証】V18 Native-Clean (Day 21 実績 0.18 / Colab 66.7%) vs V19 Precision-Coupled (ペア整合・改行保持強化)
# ⚠️ 重大教訓 (Ref 58179275 Score 0.00 の原因):
# 本番環境 (vLLM tool_call_parser="gemma4") はネイティブ <tool_call> トークンを使用します。
# 本番提出用の prompts/system.md にフォーマット指示を書くと 0.00 になります。
# したがって、本番用プロンプト本文にはフォーマット指示を一切含めず、Colab 推論アダプタ側でのみ付与します。

COLAB_SIM_ADAPTER = """
# [COLAB LOCAL SIMULATOR ADAPTER ONLY - NEVER INCLUDED IN KAGGLE SUBMISSION ZIP]
Emit ONE tool call per turn inside a ```python code block using these exact keyword argument names:
- run_command(command="...")
- read_file(path="...", start_line=1, end_line=60)
- edit_file(path="...", old_string="...", new_string="...")
- get_status()
- submit_patch()
"""

# --- [Variant 1: V18_Native_Clean (Day 21 スコア 0.18 獲得・Colab 66.7% 実証済み基準モデル)] ---
PROMPT_V18_NATIVE_CLEAN_BODY = """You are an autonomous software engineer. Your job: make the smallest, cleanest change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The exact names, messages, types and behaviours in the issue text are the specification.

# HARD LIMITS & TIME BUDGET
- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.
- CALL 7 HARD EDIT TRIGGER: By your 7th tool call, you MUST execute `edit_file`. Spend Calls 1-5 locating and reading context (primary file and call sites). By Call 6-7, apply your targeted fix. Never keep exploring past Call 7 without editing.
- ZERO CODE ECHOING: Do NOT quote, restate, or summarize code you read. State at most 1 short sentence of intent before calling the next tool.
- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`, `pytest.ini`, `conftest.py`). This is an automatic failure.
- Scratch files go in /tmp only. Never look outside /workspace. Never use code graph tools (`search_similar_code`, `get_code_neighbors`, `get_code_subgraph`).

# PRECISION READ-BEFORE-EDIT & RECOVERY PROTOCOL
1. LOCATE ONCE: Find exact file and line numbers with `run_command` using `git grep -n "KEYWORD" -- '*.py' | head -20`.
2. IMMEDIATE PINPOINT READ: As soon as grep outputs a candidate file and line N, do NOT run another grep. Immediately call `read_file` on `start_line=max(1, N-15)` to `end_line=N+35`. If caller context is needed, read it before Call 6.
3. EXACT INDENTATION COPY: In `edit_file`, copy 3 to 5 lines directly from the `read_file` output into `old_string` (without line numbers) so whitespace and indentation match character-for-character.
4. SELF-CORRECTION ON MISMATCH: If `edit_file` returns `old_string not found`, immediately call `read_file` on that exact range to check indentation, then re-apply `edit_file`. If `edit_file` fails twice, apply the exact replacement via `run_command` using `python3 -c`.
5. PARALLEL & COUPLED EDITS: If the behaviour exists in both sync and async variants (`Client`/`AsyncClient`, `def`/`async def`), update both. If you change a signature or dataclass, update its immediate consumers (<= 3 files).
6. FOCUSED VERIFY & FEEDBACK LOOP: Run a single focused test (`python -m pytest tests/path/test_x.py -x -q -k keyword`). If it passes, call `submit_patch` immediately. If it fails with an error or assertion related to your edit, read the traceback line, apply a targeted adjustment with `edit_file`, re-verify once, and call `submit_patch`.
7. FINAL CLEAN CHECK: Before calling `submit_patch`, run `git status --short && git diff` to ensure `tests/` and untracked files are untouched and `patch_size > 0`.
"""

# --- [Variant 2: V19_Precision_Coupled (Day 22 改善候補: 対称ペア更新 + 改行・境界保持強化)] ---
PROMPT_V19_PRECISION_COUPLED_BODY = """You are an autonomous software engineer. Your job: make the smallest, cleanest change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The exact names, messages, types and behaviours in the issue text are the specification.

# HARD LIMITS & TIME BUDGET
- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.
- CALL 7 HARD EDIT TRIGGER: By your 7th tool call, you MUST execute `edit_file`. Spend Calls 1-5 locating and reading context (primary file and call sites). By Call 6-7, apply your targeted fix. Never keep exploring past Call 7 without editing.
- ZERO CODE ECHOING: Do NOT quote, restate, or summarize code you read. State at most 1 short sentence of intent before calling the next tool.
- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`, `pytest.ini`, `conftest.py`). This is an automatic failure.
- Scratch files go in /tmp only. Never look outside /workspace. Never use code graph tools (`search_similar_code`, `get_code_neighbors`, `get_code_subgraph`).

# PRECISION READ-BEFORE-EDIT & RECOVERY PROTOCOL
1. LOCATE ONCE: Find exact file and line numbers with `run_command` using `git grep -n "KEYWORD" -- '*.py' | head -20`.
2. IMMEDIATE PINPOINT READ: As soon as grep outputs a candidate file and line N, do NOT run another grep. Immediately call `read_file` on `start_line=max(1, N-15)` to `end_line=N+35`. If caller context is needed, read it before Call 6.
3. EXACT INDENTATION COPY: In `edit_file`, copy 3 to 5 lines directly from the `read_file` output into `old_string` (without line numbers) so whitespace and indentation match character-for-character.
4. SELF-CORRECTION ON MISMATCH: If `edit_file` returns `old_string not found`, immediately call `read_file` on that exact range to check indentation, then re-apply `edit_file`. If `edit_file` fails twice, apply the exact replacement via `run_command` using `python3 -c`.
5. SYMMETRICAL & COUPLED EDITS: If the behaviour exists in paired attributes (title and subtitle, width and height) or sync and async variants (Client and AsyncClient, def and async def), update both symmetrically so no half-applied style or logic remains.
6. DELIMITER PRESERVATION: When preserving newlines or raw formatting, look for implicit string truncation or stripping (such as splitlines or strip) and preserve delimiters using keepends or lookahead regex splits.
7. FOCUSED VERIFY & FEEDBACK LOOP: Run a single focused test (`python -m pytest tests/path/test_x.py -x -q -k keyword`). If it passes, call `submit_patch` immediately. If it fails with an error or assertion related to your edit, read the traceback line, apply a targeted adjustment with `edit_file`, re-verify once, and call `submit_patch`.
8. FINAL CLEAN CHECK: Before calling `submit_patch`, run `git status --short && git diff` to ensure `tests/` and untracked files are untouched and `patch_size > 0`.
"""

VARIANTS = {
    "V18_Native_Clean (Day 21 実績 0.18)": PROMPT_V18_NATIVE_CLEAN_BODY + COLAB_SIM_ADAPTER,
    "V19_Precision_Coupled (Day 22 改善候補)": PROMPT_V19_PRECISION_COUPLED_BODY + COLAB_SIM_ADAPTER
}
print("[Prompt] [OK] 純粋 ADK ネイティブ準拠 2 バリアント定義完了:", list(VARIANTS.keys()))



In [ ]:
# 7. クリーンな Rich 実タスクデータセット (公式 test_patch 完備・6タスク拡張版) の取得
import urllib.request
import json
import os

CLEAN_BENCH_FILE = "/content/clean_rich_benchmark.json"
CLEAN_BENCH_URL = "https://raw.githubusercontent.com/chiba244qq/gemma4-agent-dashboard/main/data/locbench/clean_rich_benchmark.json"

print("[LocBench] GitHub から最新の clean_rich_benchmark.json (6タスク拡張版) を同期中...")
urllib.request.urlretrieve(CLEAN_BENCH_URL, CLEAN_BENCH_FILE)

with open(CLEAN_BENCH_FILE, "r", encoding="utf-8") as f:
    bench_data = json.load(f)

SELECTED_TASKS = bench_data.get("records", [])
print(f"[LocBench] [OK] 読み込み完了: 公式 test_patch 完備の実タスク {len(SELECTED_TASKS)} 件")
for t in SELECTED_TASKS:
    ps = t.get("problem_statement") or t.get("problem_statement_excerpt", "")
    print(f"  - {t['instance_id']}: tests={t.get('verification_test_files')} | ps={repr(ps[:65])}...")


In [ ]:
# 8. 【マルチターン E2E 自律実行 ＆ True-Grader v9.4 (公式 test_patch 適用絶対保証・切り捨て撤廃)】
import subprocess
import json
import os
import time

if "REPO_DIR" not in globals():
    SANDBOX_DIR = "/content/mini_kaggle_sandbox"
    REPO_DIR = os.path.join(SANDBOX_DIR, "rich_repo")
    os.makedirs(REPO_DIR, exist_ok=True)

def evaluate_task_variant(task, variant_name, system_prompt, max_turns=10):
    print(f"\n{'='*70}")
    print(f"▶ Task: {task['instance_id']} | Variant: {variant_name}")
    print(f"{'='*70}")
    
    subprocess.run(["git", "checkout", "-f", task["base_commit"]], cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "clean", "-fd"], cwd=REPO_DIR, capture_output=True)
    
    ps_text = task.get("problem_statement") or task.get("problem_statement_excerpt") or "Fix the bug in the codebase."
    issue_text = f"Issue Description for {task['instance_id']}:\n{ps_text}\n"
    if task.get("verification_test_files"):
        issue_text += f"\nRelated test files to verify: {', '.join(task['verification_test_files'])}"
        
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": issue_text}
    ]
    
    total_latency = 0.0
    parse_errors = 0
    missing_old_str_count = 0
    patch_applied = False
    prev_call_sig = None
    repeat_count = 0
    trajectory = []
    
    for turn in range(1, max_turns + 1):
        if turn == max_turns and not patch_applied:
            messages.append({"role": "user", "content": f"[SYSTEM REMINDER: Final turn {turn}/{max_turns} reached without an applied patch. You MUST call edit_file now using the exact lines from read_file.]"})
            
        cur_temp = 0.7 if repeat_count > 0 else 0.2
        resp, lat = call_llm(messages, temperature=cur_temp, max_new_tokens=1024)
        total_latency += lat
        print(f"  [Turn {turn}/{max_turns}] 推論所要: {lat:.2f}s (T={cur_temp}) | 出力長: {len(resp)} 文字")
        
        tool_call, audit = parse_tool_call(resp)
        if audit.get("missing_old_string"):
            missing_old_str_count += 1
        if not audit.get("parse_ok", True):
            parse_errors += 1
            
        if not tool_call:
            print(f"    [WARN] ツール呼出なし: {resp[:100]}...")
            trajectory.append({
                "turn": turn,
                "tool": "NONE",
                "args": "",
                "result": f"No tool call parsed. Raw head: {resp[:120]}",
                "latency_sec": round(lat, 2)
            })
            messages.append({"role": "assistant", "content": resp if resp else '(thinking)'})
            messages.append({"role": "user", "content": "Emit ONLY ONE concrete tool call inside a ```python block (e.g. run_command, read_file, or edit_file) with real arguments."})
            repeat_count += 1
            continue
            
        t_name = tool_call.get("name", "")
        t_args = tool_call.get("arguments", {})
        if not isinstance(t_args, dict):
            t_args = {"raw_input": str(t_args)}
            
        if t_name == "run_command":
            arg_summary = f"cmd={repr(str(t_args.get('command', ''))[:80])}"
        elif t_name == "read_file":
            arg_summary = f"path={repr(t_args.get('path'))}, lines={t_args.get('start_line', 1)}..{t_args.get('end_line', 'end')}"
        elif t_name == "edit_file":
            old_preview = str(t_args.get('old_string', '')).strip().splitlines()[0][:40] if t_args.get('old_string') else ''
            arg_summary = f"path={repr(t_args.get('path'))}, old={repr(old_preview)}"
        else:
            arg_summary = str(list(t_args.keys()))
        print(f"    [TOOL] Tool Call: {t_name}({arg_summary})")
        
        if t_name == "submit_patch":
            chk_diff = subprocess.run(["git", "diff"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
            if not chk_diff and not patch_applied:
                msg_reject = "Error: Cannot submit_patch because working tree has NO edits yet! You must first locate the target file with run_command(command='git grep -n ...'), inspect lines with read_file, and modify the source code with edit_file."
                print(f"    [REJECT] 未編集での submit_patch をブロックしました。探索を継続させます。")
                trajectory.append({
                    "turn": turn,
                    "tool": "submit_patch (REJECTED_EMPTY)",
                    "args": "",
                    "result": msg_reject,
                    "latency_sec": round(lat, 2)
                })
                messages.append({"role": "assistant", "content": resp})
                messages.append({"role": "user", "content": msg_reject})
                repeat_count += 1
                continue
            print("    [SUBMIT] パッチ適用済みの submit_patch が呼び出されました。探索を終了します。")
            trajectory.append({
                "turn": turn,
                "tool": "submit_patch",
                "args": "",
                "result": "Submitted patch",
                "latency_sec": round(lat, 2)
            })
            break
            
        tool_out = execute_tool(t_name, t_args, REPO_DIR)
        out_preview = tool_out[:140].strip().replace("\n", " ")
        print(f"      -> Result: {out_preview}...")
        if t_name == "edit_file" and "Successfully edited" in tool_out:
            patch_applied = True
            
        trajectory.append({
            "turn": turn,
            "tool": t_name,
            "args": arg_summary,
            "result": tool_out[:250].strip(),
            "latency_sec": round(lat, 2)
        })
            
        cur_call_sig = f"{t_name}:{json.dumps(t_args, sort_keys=True)}"
        if cur_call_sig == prev_call_sig:
            repeat_count += 1
            tool_out += "\n\n[SYSTEM WARNING: You repeated the exact same tool call! Change your arguments or move to the next step: if you ran grep, call read_file now; if you read the file, call edit_file with exact lines now.]"
        else:
            repeat_count = 0
        prev_call_sig = cur_call_sig
            
        messages.append({"role": "assistant", "content": resp})
        messages.append({"role": "user", "content": f"Tool output:\n{tool_out}"})
    
    agent_diff_res = subprocess.run(["git", "diff"], cwd=REPO_DIR, capture_output=True, text=True)
    agent_diff = agent_diff_res.stdout.strip()
    has_diff = len(agent_diff) > 0
    
    pytest_pass = False
    pytest_log = ""
    test_patch_applied = False
    test_patch = task.get("test_patch", "")
    test_files = task.get("verification_test_files", [])
    
    if not has_diff:
        verdict = "[FAIL] NO_PATCH (Score 0)"
    else:
        if test_patch:
            patch_path = "/tmp/official_test_patch.diff"
            patch_content = test_patch if test_patch.endswith("\n") else (test_patch + "\n")
            with open(patch_path, "w", encoding="utf-8") as pf:
                pf.write(patch_content)
            apply_res = subprocess.run(["git", "apply", "--whitespace=nowarn", patch_path], cwd=REPO_DIR, capture_output=True, text=True)
            if apply_res.returncode != 0:
                test_patch_applied = False
                verdict = f"[ERROR] TEST_PATCH_APPLY_FAILED: {apply_res.stderr.strip()[:80]}"
                print(f"    [Grader ERROR] test_patch 適用失敗: {apply_res.stderr.strip()[:100]}")
            else:
                test_patch_applied = True
                print("    [Grader] [OK] 公式 test_patch をワーキングツリーへ適用完了")
        else:
            test_patch_applied = False
            
        if test_patch_applied:
            target_tests = test_files if test_files else ["tests/"]
            cmd = ["pytest", "-q", "--maxfail=1"] + target_tests
            py_res = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, timeout=60)
            pytest_raw_pass = (py_res.returncode == 0)
            pytest_log = (py_res.stdout + "\n" + py_res.stderr)[-500:].strip()
            pytest_pass = pytest_raw_pass
            verdict = "[TRUE_PASS] PASS" if pytest_pass else "[FAIL] FAIL"
        else:
            pytest_pass = False
            pytest_log = f"ERROR: Official test_patch was not applied ({verdict}). Cannot grade."
        
    print(f"  [REPORT] Task 判定: パッチ生成={has_diff} | test_patch適用={test_patch_applied} | 真の合否={verdict}")
    if pytest_log:
        print(f"    [pytest log summary]:\n{pytest_log[:300]}")
        
    subprocess.run(["git", "checkout", "-f", task["base_commit"]], cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "clean", "-fd"], cwd=REPO_DIR, capture_output=True)
    
    return {
        "task_id": task["instance_id"],
        "variant": variant_name,
        "pytest_pass": pytest_pass,
        "has_diff": has_diff,
        "test_patch_applied": test_patch_applied,
        "turns_used": turn,
        "total_latency_sec": round(total_latency, 2),
        "avg_step_latency_sec": round(total_latency / max(1, turn), 2),
        "parse_errors": parse_errors,
        "missing_old_str_count": missing_old_str_count,
        "agent_diff": agent_diff,
        "trajectory": trajectory,
        "pytest_log": pytest_log
    }

all_eval_results = []
for t in SELECTED_TASKS:
    for v_name, v_prompt in VARIANTS.items():
        res = evaluate_task_variant(t, v_name, v_prompt, max_turns=10)
        all_eval_results.append(res)
print("\n[DONE] 全 A/B 実機検証ラン (v9.4 公式 test_patch 完全保証版) が完了しました！")


In [ ]:
# 9. 結果集計・統計比較表の出力 & GitHub 自動同期
import pandas as pd
from tabulate import tabulate
import time

df = pd.DataFrame(all_eval_results)
summary = []
for v_name in VARIANTS.keys():
    v_df = df[df["variant"] == v_name]
    n = len(v_df)
    if n == 0:
        continue
    pass_rate = (v_df["pytest_pass"].sum() / n) * 100
    patch_rate = (v_df["has_diff"].sum() / n) * 100
    tp_applied_rate = (v_df["test_patch_applied"].sum() / n) * 100 if "test_patch_applied" in v_df.columns else 0.0
    avg_latency = v_df["avg_step_latency_sec"].mean()
    avg_turns = v_df["turns_used"].mean()
    total_parse_err = v_df["parse_errors"].sum()
    total_missing_old = v_df["missing_old_str_count"].sum()
    
    summary.append({
        "Variant": v_name,
        "Tasks": n,
        "True Pass Rate (%)": f"{pass_rate:.1f}%",
        "Test Patch OK (%)": f"{tp_applied_rate:.1f}%",
        "Patch Rate (%)": f"{patch_rate:.1f}%",
        "Parse Errors": int(total_parse_err),
        "Missing old_string": int(total_missing_old),
        "Avg Step Latency": f"{avg_latency:.2f}s",
        "Avg Turns": f"{avg_turns:.1f}"
    })

sum_df = pd.DataFrame(summary)
print("="*95)
print("🏆 【クリーン実タスク (Rich) A/B 検証結果サマリー (v9.4 公式 test_patch 適用保証版)】")
print("="*95)
print(tabulate(sum_df, headers="keys", tablefmt="github", showindex=False))

now_str = time.strftime("%Y-%m-%d %H:%M:%S JST")
output_data = {
    "timestamp": now_str,
    "model": MODEL_ID,
    "hardware_notes": LOAD_NOTES if "LOAD_NOTES" in globals() else "",
    "eval_mode": "clean_rich_tasks",
    "grader_version": "v9.4_enforced_test_patch",
    "summary": summary,
    "detailed_results": all_eval_results
}

try:
    github_pat = userdata.get('GITHUB_PAT')
    if github_pat:
        import requests, base64
        repo = "chiba244qq/gemma4-agent-dashboard"
        path = "data/locbench_ab_test_results.json"
        api_url = f"https://api.github.com/repos/{repo}/contents/{path}"
        headers = {"Authorization": f"token {github_pat}", "Accept": "application/vnd.github.v3+json"}
        sha = None
        get_res = requests.get(api_url, headers=headers)
        if get_res.status_code == 200:
            sha = get_res.json().get("sha")
        content_bytes = json.dumps(output_data, indent=2, ensure_ascii=False).encode('utf-8')
        payload = {
            "message": f"[LocBench A/B Sync] Rich クリーンタスク評価結果更新 ({now_str})",
            "content": base64.b64encode(content_bytes).decode('utf-8')
        }
        if sha:
            payload["sha"] = sha
        put_res = requests.put(api_url, headers=headers, json=payload)
        if put_res.status_code in [200, 201]:
            print(f"\n[Sync] 🎉 GitHub ({path}) への自動プッシュに成功しました！")
        else:
            print(f"\n[Sync Warning] GitHub プッシュ失敗: {put_res.status_code}")
except Exception as e:
    print(f"\n[Sync Notice] GitHub 同期スキップ: {e}")

# ------------------------------------------------------------------------------
# 🔌 全タスク検証完了後の自動インスタンス切断 (CU消費防止)
# ------------------------------------------------------------------------------
AUTO_DISCONNECT_ON_FINISH = True
if AUTO_DISCONNECT_ON_FINISH:
    print("\n" + "="*80)
    print("🔌 [Auto-Disconnect] 全検証および GitHub 同期が完了しました。無駄な CU 消費を防ぐためインスタンスを切断 (runtime.unassign) します...")
    print("="*80)
    time.sleep(3)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"[Notice] runtime.unassign() スキップ: {e}")
